# AI Practice Notebook

Solved practice questions for:
- BFS and DFS
- A*
- Hill Climbing
- Minimax
- Alpha-Beta Pruning
- Genetic Algorithm

Each section prints a concrete result so you can directly verify behavior.

In [1]:
from collections import deque
import heapq
import math
import random

## 1) BFS and DFS

Practice Question: Given the graph, find a path from A to G using BFS and DFS.

In [2]:
graph_unweighted = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': ['G'],
    'F': ['G'],
    'G': []
}

def bfs_path(graph, start, goal):
    q = deque([(start, [start])])
    visited = {start}
    order = []
    while q:
        node, path = q.popleft()
        order.append(node)
        if node == goal:
            return path, order
        for nbr in graph[node]:
            if nbr not in visited:
                visited.add(nbr)
                q.append((nbr, path + [nbr]))
    return None, order

def dfs_path(graph, start, goal):
    stk = [(start, [start])]
    visited = set()
    order = []
    while stk:
        node, path = stk.pop()
        if node in visited:
            continue
        visited.add(node)
        order.append(node)
        if node == goal:
            return path, order
        for nbr in reversed(graph[node]):
            if nbr not in visited:
                stk.append((nbr, path + [nbr]))
    return None, order

bfs_res, bfs_order = bfs_path(graph_unweighted, 'A', 'G')
dfs_res, dfs_order = dfs_path(graph_unweighted, 'A', 'G')

print('BFS visit order:', bfs_order)
print('BFS path A->G  :', bfs_res)
print('DFS visit order:', dfs_order)
print('DFS path A->G  :', dfs_res)

BFS visit order: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
BFS path A->G  : ['A', 'B', 'E', 'G']
DFS visit order: ['A', 'B', 'D', 'E', 'G']
DFS path A->G  : ['A', 'B', 'E', 'G']


## 2) A* Search

Practice Question: Find the least-cost path from A to G using A* and heuristic values.

In [3]:
graph_weighted = {
    'A': [('B', 1), ('C', 4)],
    'B': [('D', 2), ('E', 5)],
    'C': [('F', 1)],
    'D': [('G', 5)],
    'E': [('G', 1)],
    'F': [('G', 3)],
    'G': []
}
heuristic = {'A': 6, 'B': 4, 'C': 4, 'D': 3, 'E': 1, 'F': 2, 'G': 0}

def astar(graph, h, start, goal):
    pq = [(h[start], 0, start, [start])]
    best_g = {}
    expanded = []
    while pq:
        f, g, node, path = heapq.heappop(pq)
        if node in best_g and best_g[node] <= g:
            continue
        best_g[node] = g
        expanded.append((node, g, h[node], f))
        if node == goal:
            return path, g, expanded
        for nbr, w in graph[node]:
            ng = g + w
            nf = ng + h[nbr]
            if nbr not in best_g or ng < best_g[nbr]:
                heapq.heappush(pq, (nf, ng, nbr, path + [nbr]))
    return None, math.inf, expanded

path, cost, trace = astar(graph_weighted, heuristic, 'A', 'G')
print('A* expansion trace (node, g, h, f):')
for row in trace:
    print(row)
print('Optimal path:', path)
print('Total cost :', cost)

A* expansion trace (node, g, h, f):
('A', 0, 6, 6)
('B', 1, 4, 5)
('D', 3, 3, 6)
('E', 6, 1, 7)
('G', 7, 0, 7)
Optimal path: ['A', 'B', 'E', 'G']
Total cost : 7


## 3) Hill Climbing

Practice Question: Maximize the objective function by moving to the best neighbor.

In [4]:
def objective(x):
    return -(x - 4) ** 2 + 16

def hill_climb(start, low=0, high=8):
    x = start
    route = [x]
    while True:
        nbrs = []
        if x - 1 >= low:
            nbrs.append(x - 1)
        if x + 1 <= high:
            nbrs.append(x + 1)
        best = max(nbrs, key=objective)
        if objective(best) <= objective(x):
            break
        x = best
        route.append(x)
    return x, objective(x), route

best_x, best_val, route = hill_climb(start=0)
print('Visited states:', route)
print('Final state   :', best_x)
print('Objective     :', best_val)

Visited states: [0, 1, 2, 3, 4]
Final state   : 4
Objective     : 16


## 4) Minimax

Practice Question: Evaluate a game tree and pick best move for MAX.

In [5]:
# Tree format: [left_subtree, right_subtree] with integer leaves
game_tree = [[[3, 5], [6, 9]], [[1, 2], [0, -1]]]

def minimax(tree, maximizing):
    if isinstance(tree, int):
        return tree
    vals = [minimax(child, not maximizing) for child in tree]
    return max(vals) if maximizing else min(vals)

root_value = minimax(game_tree, True)
left_value = minimax(game_tree[0], False)
right_value = minimax(game_tree[1], False)
best_move = 'Left' if left_value >= right_value else 'Right'

print('Left branch value :', left_value)
print('Right branch value:', right_value)
print('Root minimax value:', root_value)
print('Best move for MAX :', best_move)

Left branch value : 5
Right branch value: 0
Root minimax value: 5
Best move for MAX : Left


## 5) Alpha-Beta Pruning

Practice Question: Compute same game value as Minimax but with pruning.

In [6]:
def alphabeta(tree, alpha, beta, maximizing, counter):
    if isinstance(tree, int):
        counter['leaves'] += 1
        return tree

    if maximizing:
        value = -math.inf
        for child in tree:
            value = max(value, alphabeta(child, alpha, beta, False, counter))
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value

    value = math.inf
    for child in tree:
        value = min(value, alphabeta(child, alpha, beta, True, counter))
        beta = min(beta, value)
        if alpha >= beta:
            break
    return value

counter = {'leaves': 0}
ab_value = alphabeta(game_tree, -math.inf, math.inf, True, counter)
mm_value = minimax(game_tree, True)

print('Alpha-Beta value      :', ab_value)
print('Minimax value         :', mm_value)
print('Leaves evaluated (AB) :', counter['leaves'])

Alpha-Beta value      : 5
Minimax value         : 5
Leaves evaluated (AB) : 5


## 6) Genetic Algorithm

Practice Question: Maximize $f(x) = x^2$ for 4-bit chromosome in range 0..15.

In [7]:
random.seed(11)

def decode(bits):
    return int(bits, 2)

def fitness(bits):
    x = decode(bits)
    return x * x

def select_top2(pop):
    scored = sorted(pop, key=fitness, reverse=True)
    return scored[:2], scored

def crossover(a, b, cut=2):
    return a[:cut] + b[cut:], b[:cut] + a[cut:]

def mutate(bits, p=0.1):
    arr = list(bits)
    for i in range(len(arr)):
        if random.random() < p:
            arr[i] = '1' if arr[i] == '0' else '0'
    return ''.join(arr)

population = ['0011', '0101', '1001', '1110']
print('Initial population:', population)

for gen in range(1, 6):
    parents, ranked = select_top2(population)
    c1, c2 = crossover(parents[0], parents[1], cut=2)
    c1, c2 = mutate(c1, p=0.15), mutate(c2, p=0.15)
    population = [parents[0], parents[1], c1, c2]
    best = max(population, key=fitness)
    print(f'Gen {gen}: best={best}, x={decode(best)}, fit={fitness(best)}')

best_overall = max(population, key=fitness)
print('Final best chromosome:', best_overall)
print('Final best x         :', decode(best_overall))
print('Final best fitness   :', fitness(best_overall))

Initial population: ['0011', '0101', '1001', '1110']
Gen 1: best=1110, x=14, fit=196
Gen 2: best=1111, x=15, fit=225
Gen 3: best=1111, x=15, fit=225
Gen 4: best=1111, x=15, fit=225
Gen 5: best=1111, x=15, fit=225
Final best chromosome: 1111
Final best x         : 15
Final best fitness   : 225


## 7) Generic Model Template

Use this checklist for any AI technique:
1. Define state, start, actions, transition, goal.
2. Define evaluation: cost, heuristic, utility, or fitness.
3. Choose selection strategy: queue/stack/best-first/local/evolutionary.
4. Expand and update memory (visited/open/population).
5. Stop on goal/terminal/convergence and report decision path.

In [8]:
def bfs(graph, start, end):
    q = deque([(start, [start])])
    visited = [start]
    seen = set([start])
    while q:
        node, path = q.popleft()
        if(node == end):
            return path, visited
        for nextNode in graph[node]:
            if nextNode not in seen:
                seen.add(nextNode)
                visited.append(nextNode)
                q.append((nextNode, path + [nextNode]))
    return None, visited

city_map = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F', 'G'],
    'F': ['C', 'E'],
    'G': ['E']
}

path, visited = bfs(city_map, 'A', 'G')
print('Visited cities in order:', visited)
print('Path from A to G:', path)

Visited cities in order: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Path from A to G: ['A', 'B', 'E', 'G']


In [9]:
def dfsMaze(maze, start, end):
    rows, cols = len(maze), len(maze[0])
    stk = [(start,[start])]
    visited = set([start])
    moves = [(0,1),(0,-1),(1,0),(-1,0)]
    while stk:
        (x,y), path = stk.pop()
        if(x,y) == end:
            return path
        for dx, dy in moves:
            nx, ny = x + dx, y + dy
            if(0 <= nx < rows and 0 <= ny < cols and maze[nx][ny] == 0 and (nx, ny) not in visited):
                visited.add((nx, ny))
                stk.append(((nx, ny), path + [(nx, ny)]))
    return None
maze = [
    [0, 0, 1, 0],
    [1, 0, 1, 0],
    [0, 0, 0, 1],
    [0, 1, 0, 0]
]
start = (0, 0)
end = (3, 3)
path = dfsMaze(maze, start, end)
print('Path from start to end:', path)
def bfsMaze(maze, start, end):
    rows, cols = len(maze), len(maze[0])
    q = deque([(start,[start])])
    visited = set([start])
    moves = [(0,1),(0,-1),(1,0),(-1,0)]
    while q:
        (x,y), path = q.popleft()
        if(x,y) == end:
            return path
        for dx, dy in moves:
            nx, ny = x + dx, y + dy
            if(0 <= nx < rows and 0 <= ny < cols and maze[nx][ny] == 0 and (nx, ny) not in visited):
                visited.add((nx, ny))
                q.append(((nx, ny), path + [(nx, ny)]))
    return None
maze = [
    [0, 0, 1, 0],
    [1, 0, 1, 0],
    [0, 0, 0, 1],
    [0, 1, 0, 0]
]
start = (0, 0)
end = (3, 3)
path = bfsMaze(maze, start, end)
print('Path from start to end:', path)

Path from start to end: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (3, 2), (3, 3)]
Path from start to end: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (3, 2), (3, 3)]


In [11]:
def ucsLowestCost(graph, start, end):
    pq = [(0, start, [start])]
    visited = set([start])
    while pq:
        cost, node, path = heapq.heappop(pq)
        if node == end:
            return path, cost
        for nextNode, edgeCost in graph[node]:
            if nextNode not in visited:
                visited.add(nextNode)
                heapq.heappush(pq, (cost + edgeCost, nextNode, path + [nextNode]))
    return None, float('inf')
weighted_graph = {
    'A': [('B', 1), ('C', 4)],
    'B': [('D', 2), ('E', 5)],
    'C': [('F', 1)],
    'D': [('G', 5)],
    'E': [('G', 1)],
    'F': [('G', 3)],
    'G': []
}
path, cost = ucsLowestCost(weighted_graph, 'A', 'G')
print('Lowest cost path from A to G:', path)
print('Total cost of the path:', cost)


Lowest cost path from A to G: ['A', 'B', 'D', 'G']
Total cost of the path: 8


In [12]:
def depth_limited_dfs(tree, node, target, limit, depth=0):
    if node == target:
        return [node]
    if depth >= limit:
        return None
    if node in tree:
        for child in tree[node]:
            result = depth_limited_dfs(tree, child, target, limit, depth + 1)
            if result:
                return [node] + result
    return None

def iterative_deepening_dfs(tree, start, target, max_depth):
    for depth in range(max_depth + 1):
        result = depth_limited_dfs(tree, start, target, depth)
        if result:
            return result
    return None

tree = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': ['G'],
    'F': ['G'],
    'G': []
}
path = iterative_deepening_dfs(tree, 'A', 'G', max_depth=3)
print('Path from A to G using IDDFS:', path)


Path from A to G using IDDFS: ['A', 'B', 'E', 'G']


In [14]:

graph = {
    'A': [('B', 2), ('C', 3)],
    'B': [('D', 4), ('E', 5)],
    'C': [('F', 6)],
    'D': [('G', 7)],
    'E': [('G', 3)],
    'F': [('G', 8)],
    'G': []
}

heuristic = {
    'A': 6, 'B': 4, 'C': 5,
    'D': 2, 'E': 3, 'F': 6, 'G': 0
}

def greedyBFS(graph, heuristic, start, goal):
    pq = [(heuristic[start], start, [start])]
    visited = set([start])
    excpanded = []
    while pq:
        huiristic_value, node, path = heapq.heappop(pq)
        excpanded.append((node, heuristic[node]))
        if node == goal:
            return path, excpanded
        for nextNode, cost in graph[node]:
            if nextNode not in visited:
                visited.add(nextNode)
                heapq.heappush(pq, (heuristic[nextNode], nextNode, path + [nextNode]))
    return None, excpanded
path, expanded = greedyBFS(graph, heuristic, 'A', 'G')
print('Expanded nodes (node, heuristic):')
for node in expanded:
    print(node)
print('Path from A to G using Greedy BFS:', path)

Expanded nodes (node, heuristic):
('A', 6)
('B', 4)
('D', 2)
('G', 0)
Path from A to G using Greedy BFS: ['A', 'B', 'D', 'G']


In [16]:
graph = {
    'A': [('B', 2), ('C', 3)],
    'B': [('D', 4), ('E', 5)],
    'C': [('F', 6)],
    'D': [('G', 7)],
    'E': [('G', 3)],
    'F': [('G', 8)],
    'G': []
}

heuristic = {
    'A': 6, 'B': 4, 'C': 5,
    'D': 2, 'E': 3, 'F': 6, 'G': 0
}

def aStar(graph, heuristic, start, goal):
    pq = [(heuristic[start], 0, start, [start])]
    visited = {}
    expanded = []

    while pq:
        f, g, node, path = heapq.heappop(pq)
        if node in visited and visited[node] <= g:
            continue

        visited[node] = g
        expanded.append((node, g, heuristic[node], f))
        print(f"  {node:<6} {g:<8} {heuristic[node]:<8} {f:<8} {' -> '.join(path)}")

        if node == goal:
            return path, g, expanded

        for nextNode, cost in graph[node]:
            ng = g + cost
            nf = ng + heuristic[nextNode]
            if nextNode not in visited or ng < visited[nextNode]:
                heapq.heappush(pq, (nf, ng, nextNode, path + [nextNode]))

    return None, float('inf'), expanded
path, cost, trace = aStar(graph, heuristic, 'A', 'G')
print('A* expansion trace (node, g, h, f):')
for row in trace:
    print(row)
print('Optimal path from A to G:', path)
print('Total cost of the path:', cost)
        

  A      0        6        6        A
  B      2        4        6        A -> B
  C      3        5        8        A -> C
  D      6        2        8        A -> B -> D
  E      7        3        10       A -> B -> E
  G      10       0        10       A -> B -> E -> G
A* expansion trace (node, g, h, f):
('A', 0, 6, 6)
('B', 2, 4, 6)
('C', 3, 5, 8)
('D', 6, 2, 8)
('E', 7, 3, 10)
('G', 10, 0, 10)
Optimal path from A to G: ['A', 'B', 'E', 'G']
Total cost of the path: 10


In [ ]:
class Node:
  def __init__(self, board, player, move=None):
    self.board = board
    self.player = player
    self.move = move
    self.value = None
    self.children = []

def get_moves(board):
  return [i for i, v in enumerate(board) if v == ' ']

def winner(board):
  lines = [
    (0,1,2), (3,4,5), (6,7,8),
    (0,3,6), (1,4,7), (2,5,8),
    (0,4,8), (2,4,6)
  ]
  for a, b, c in lines:
    if board[a] != ' ' and board[a] == board[b] == board[c]:
      return board[a]
  return None

def is_terminal(board):
  return winner(board) is not None or ' ' not in board

def utility(board):
  w = winner(board)
  if w == 'X':
    return 1
  if w == 'O':
    return -1
  return 0

def minimax_ttt(node, is_max):
  if is_terminal(node.board):
    node.value = utility(node.board)
    return node.value

  if is_max:
    best = -999
    for move in get_moves(node.board):
      new_board = node.board[:]
      new_board[move] = 'X'
      child = Node(new_board, 'O', move)
      node.children.append(child)
      best = max(best, minimax_ttt(child, False))
    node.value = best
    return best
  else:
    best = 999
    for move in get_moves(node.board):
      new_board = node.board[:]
      new_board[move] = 'O'
      child = Node(new_board, 'X', move)
      node.children.append(child)
      best = min(best, minimax_ttt(child, True))
    node.value = best
    return best

def get_best_move(board):
  root = Node(board, 'X')
  best_move = None
  best_value = -999

  print("Evaluating moves:")
  for move in get_moves(board):
    new_board = board[:]
    new_board[move] = 'X'
    child = Node(new_board, 'O', move)
    root.children.append(child)
    value = minimax_ttt(child, False)
    child.value = value
    print(f"  Index {move} -> value = {value}")
    if value > best_value:
      best_value = value
      best_move = move

  root.value = best_value
  return best_move, best_value, root

NameError: name 'example_tree' is not defined

In [7]:
def alphabetaPrune(tree, alpha, beta, isMax, path="Root"):
    if isinstance(tree, int):
        return tree, path

    if isMax:
        value = -math.inf
        best_path = path
        for child in tree:
            childValue, childPath = alphabetaPrune(child, alpha, beta, False, path + f" -> {child}")
            if childValue > value:
                value = childValue
                best_path = childPath
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value, best_path
    else:
        value = math.inf
        best_path = path
        for child in tree:
            childValue, childPath = alphabetaPrune(child, alpha, beta, True, path + f" -> {child}")
            if childValue < value:
                value = childValue
                best_path = childPath
            beta = min(beta, value)
            if alpha >= beta:
                break
        return value, best_path

ab_value, ab_path = alphabetaPrune(game_tree, -math.inf, math.inf, True)

print('Alpha-Beta value      :', ab_value)
print('Alpha-Beta path       :', ab_path)

Alpha-Beta value      : 5
Alpha-Beta path       : Root -> [[3, 5], [6, 9]] -> [3, 5] -> 5


In [8]:
def elevation(x):
    return -(x - 4) ** 2 + 16

x = 0
route = [x]
while True:
    Neighbors = []
    if x - 1 >= 0:
        Neighbors.append(x - 1)
    if x + 1 <= 8:
        Neighbors.append(x + 1)
    best = max(Neighbors, key=elevation)
    if elevation(best) <= elevation(x):
        break
    x = best
    route.append(x)
print('Visited states:', route)
print('Final state   :', x)
print('Elevation     :', elevation(x))

Visited states: [0, 1, 2, 3, 4]
Final state   : 4
Elevation     : 16


In [9]:
productivity = {'A': 3, 'B': 6, 'C': 5, 'D': 7, 'E': 9, 'F': 8}
neighbors = {'A': ['B', 'C'], 'B': ['D'], 'C': ['D'], 'D': ['E', 'F'], 'E': [], 'F': []}

current = 'A'
path = [current]

while True:
    nbrs = neighbors[current]
    if not nbrs:
        break
    best = max(nbrs, key=lambda s: productivity[s])
    if productivity[best] <= productivity[current]:
        break
    current = best
    path.append(current)

print("States visited:", " -> ".join(path))
print(f"Final state: {current}, Productivity: {productivity[current]}")

States visited: A -> B -> D -> E
Final state: E, Productivity: 9


In [10]:
population = ["0011", "0101", "1001", "1110"]
rows = []
for ch in population:
    x = int(ch, 2)
    fit = x ** 2
    rows.append((ch, x, fit))
ranked = sorted(rows, key=lambda t: t[2], reverse=True)
parents = ranked[:2]
print("Chromosome | x | fitness")
for r in rows:
    print(r[0], r[1], r[2])
print("\nRanked (best to worst):")
for r in ranked:
    print(r)
print("\nSelected parents:")
for p in parents:
    print(p)

Chromosome | x | fitness
0011 3 9
0101 5 25
1001 9 81
1110 14 196

Ranked (best to worst):
('1110', 14, 196)
('1001', 9, 81)
('0101', 5, 25)
('0011', 3, 9)

Selected parents:
('1110', 14, 196)
('1001', 9, 81)
